In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
silver_df = spark.table(
    "finance_ml.silver.account_events_clean"
)

In [0]:
def build_gold_features(silver_df):
    """
    Engineering causal Gold features from a Silver Spark DataFrame.

    Args:
        silver_df:
            Cleaned Silver Spark DataFrame.

    Returns:
        Spark DataFrame containing original cleaned columns and engineered
        causal features.
    """
    from pyspark.sql import functions as F
    from pyspark.sql.window import Window

    ordered_window = Window.partitionBy("account_id").orderBy("date")

    rolling_7d_window = ordered_window.rowsBetween(-6, 0)

    gold_df = (
        silver_df
        # Same-day derived features.
        .withColumn(
            "net_cash_flow",
            F.col("cash_inflow") - F.col("cash_outflow"),
        )
        .withColumn(
            "payment_gap",
            F.col("amount_due") - F.col("amount_paid"),
        )
        # One-step lag features.
        .withColumn(
            "previous_balance",
            F.lag("balance", 1).over(ordered_window),
        )
        .withColumn(
            "previous_credit_utilization",
            F.lag(
                "credit_utilization",
                1,
            ).over(ordered_window),
        )
        .withColumn(
            "previous_payment_ratio",
            F.lag(
                "payment_ratio",
                1,
            ).over(ordered_window),
        )
        .withColumn(
            "previous_days_past_due",
            F.lag(
                "days_past_due",
                1,
            ).over(ordered_window),
        )
        # One-step change features.
        .withColumn(
            "balance_change",
            F.col("balance") - F.col("previous_balance"),
        )
        .withColumn(
            "credit_utilization_change",
            F.col("credit_utilization") - F.col("previous_credit_utilization"),
        )
        .withColumn(
            "payment_ratio_change",
            F.col("payment_ratio") - F.col("previous_payment_ratio"),
        )
        # Seven-day trailing rolling features.
        .withColumn(
            "rolling_7d_mean_balance",
            F.avg("balance").over(rolling_7d_window),
        )
        .withColumn(
            "rolling_7d_mean_cash_outflow",
            F.avg("cash_outflow").over(rolling_7d_window),
        )
        .withColumn(
            "rolling_7d_mean_payment_ratio",
            F.avg("payment_ratio").over(rolling_7d_window),
        )
        .withColumn(
            "rolling_7d_max_days_past_due",
            F.max("days_past_due").over(rolling_7d_window),
        )
    )

    return gold_df

In [0]:
gold_df = build_gold_features(silver_df)

In [0]:
print(gold_df.count())

In [0]:
gold_df.select(
    "account_id",
    "date",
    "balance",
    "previous_balance",
    "balance_change",
    "net_cash_flow",
    "payment_gap",
    "rolling_7d_mean_balance",
    "rolling_7d_mean_cash_outflow",
    "rolling_7d_mean_payment_ratio",
    "rolling_7d_max_days_past_due",
).show(15, truncate=False)

In [0]:
(
    gold_df.write
    .mode("overwrite")
    .saveAsTable("finance_ml.gold.account_features")
)

In [0]:
spark.table(
    "finance_ml.gold.account_features"
).count()

In [0]:
gold_persisted_df = spark.table(
    "finance_ml.gold.account_features"
)

print(gold_persisted_df.count())

gold_persisted_df.select(
    "account_id",
    "date",
    "net_cash_flow",
    "payment_gap",
    "previous_balance",
    "balance_change",
    "rolling_7d_mean_balance",
).show(10, truncate=False)